In [1]:
#%env JAX_PLATFORM=cpu
%env CUDA_VISIBLE_DEVICES=0
%env XLA_PYTHON_CLIENT_PREALLOCATE=false
%env JAX_ENABLE_X64=True
%env JAX_CHECK_TRACER_LEAKS=true

import jax
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt

env: CUDA_VISIBLE_DEVICES=0
env: XLA_PYTHON_CLIENT_PREALLOCATE=false
env: JAX_ENABLE_X64=True
env: JAX_CHECK_TRACER_LEAKS=true


In [2]:
from systems import ToyExample, FaultyDSSM, DSSM
from kalman import KalmanFilter
from mpc import MPC, References, Costs as MPCCosts
import equinox as eqx
from jaxtyping import Array, Float


class SimCosts(eqx.Module):
    y: Float[Array, "y y"]
    u: Float[Array, "u u"]
    x: Float[Array, "x x"]
    z: Float[Array, "z z"]

    def to_mpc(self):
        x = jnp.pad(self.x, ((0,self.z.shape[0]), (0, self.z.shape[1])))
        return MPCCosts(y=self.y, u=self.u, x=x)


sys = ToyExample(x_noise_std=0.01, y_noise_std=0.01)
print(sys.as_dssm())

AugmentedSystem(x=f64[6], x_dim=6, u_dim=2, y_dim=2)


In [3]:
ref = References(
    y=jnp.zeros((100, sys.y_dim)),
    u=jnp.zeros((100, sys.u_dim)),
    x=jnp.zeros((100, sys.as_dssm().x_dim))
)
J = SimCosts(
    y=1.0 * jnp.eye(sys.y_dim),
    u=0.001 * jnp.eye(sys.u_dim),
    x=0.0 * jnp.eye(sys.x_dim),
    z=1.0 * jnp.eye(sys.z_dim),
)
Q = 0.01 * jnp.eye(sys.as_dssm().x_dim)
R = 0.01 * jnp.eye(sys.y_dim)


J_mpc = MPCCosts(y=J.y, u=J.u, x=jnp.pad(J.x, ((0, sys.z_dim), (0, sys.z_dim))))

mpc = MPC(sys.as_dssm(), horizon=10, discount=0.9, ref=ref, J=J_mpc)
kf = KalmanFilter(sys.as_dssm(), Q=Q, R=R)

In [ ]:
print(sys)

ValueError: `where` does not specify an element or elements of `pytree`.

In [ ]:
class Simulation(eqx.Module):
    sys: FaultyDSSM
    mpc: MPC
    kf: KalmanFilter

    J: SimCosts = eqx.field(static=True)

    def replace(self, sys, mpc, kf):
        return eqx.tree_at(lambda s: (s.sys, s.mpc, s.kf), self, (sys, mpc, kf))

    def reset(self, *, rng=None):
        return self.replace(
            sys=self.sys.reset(rng=rng),
            kf=self.kf.reset(),
            mpc=self.mpc.reset(),
        )

    @staticmethod
    def step(sim, rng):
        rng_step, rng_act = jr.split(rng)
        mpc = sim.mpc.update(x=sim.kf.x)
        sys, y = sim.sys(mpc.u, rng=rng_step)
        kf = sim.kf.update(mpc.u, y)
        return sim.replace(sys=sys, mpc=mpc, kf=kf), y

    def rollout(self, rng):
        n = len(self.mpc.ref.y)-self.mpc.horizon
        self = self.reset(rng=rng)
        return jax.lax.scan(self.step, self, jr.split(rng, n))[1]


sim = Simulation(sys=sys, mpc=mpc, kf=kf, J=J)
print(sys.reset(rng=jr.key(0)))
print(mpc.reset())
print(kf.reset())

sim = sim.reset(rng=jr.key(0))

step = lambda s, input: s.step(*input)

sim, out = jax.lax.scan(step, sim, (jnp.arange(100), jr.split(jr.key(0), 100)))
out

ValueError: `where` does not specify an element or elements of `pytree`.

RecursionError: maximum recursion depth exceeded

In [ ]:
sim.sys.reset()

ToyExample(
  x=f64[2],
  z=f64[4],
  z_dim=4,
  x_dim=2,
  u_dim=2,
  y_dim=2,
  input_coef=1.0,
  flow_coef=0.5,
  output_coef=1.0,
  x_noise_std=0.01,
  y_noise_std=0.01
)